In [21]:
!pip install ipytest

In [22]:
import ipytest
import pytest
ipytest.autoconfig()

In [23]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

class InsuranceDataProcessor:
    def __init__(self, input_path, output_path):
        """
        Inicializa la clase con las rutas de entrada y salida.
        """
        self.input_path = input_path
        self.output_path = output_path
        self.data = None
        self.cleaned_data = None

    def load_data(self):
        """
        Carga el dataset desde la ruta especificada.
        """
        self.data = pd.read_csv(self.input_path, header=None)

    def clean_data(self):
        """
        Limpia los datos: convierte columnas object a numéricas y maneja valores nulos.
        """
        for col in self.data.columns:
            if self.data[col].dtype == 'object':
                self.data[col] = pd.to_numeric(self.data[col], errors='coerce')
                self.data[col] = self.data[col].round().astype('Int64')

        self.data.fillna(value=np.nan, inplace=True)
        nulos_pre = self.data.isnull().mean() * 100
        print("Porcentaje de nulos por columna:")
        print(nulos_pre.sort_values(ascending=False))

        # Eliminar la última columna
        last_col = self.data.columns[-1]
        self.data.drop(columns=[last_col], inplace=True)
        print(f"Última columna eliminada: {last_col}")


    def validate_target_variable(self):
        """
        Valida la variable de salida (última columna) y elimina registros inválidos.
        """
        target_col = self.data.columns[-1]

        null_count = self.data[target_col].isnull().sum()
        null_percentage = (null_count / len(self.data)) * 100
        self.data.dropna(subset=[target_col], inplace=True)
        self.data = self.data[self.data[target_col].isin([0, 1])]

    def export_data(self):
        """
        Exporta el dataset limpio a la ruta especificada.
        """
        self.data.to_csv(self.output_path, index=False)

    def load_clean_data(self):
        """
        Guarda el dataset limpio en una variable aparte.
        """
        self.cleaned_data = self.data.copy()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
@pytest.fixture
def processor():
    input_path = '/content/drive/MyDrive/Colab Notebooks/MLOps/Tarea Fase 1/insurance_company_modified.csv'  # Cambia esto por tu ruta real
    output_path = '/content/drive/My Drive/Colab Notebooks/MLOps/Tarea Fase 3/insurance_clean.csv'
    proc = InsuranceDataProcessor(input_path, output_path)
    proc.load_data()
    proc.clean_data() # Preprocesamiento antes de validar
    return proc


In [25]:
%%ipytest

'''Verifica que los datos se hayan cargado correctamente'''
def test_load_data(processor):
    assert processor.data is not None
    assert isinstance(processor.data, pd.DataFrame)
    assert not processor.data.empty

'''Verifica que se haya eliminado una columna (la última)'''
def test_clean_data(processor):
    original_cols = processor.data.shape[1]
    processor.clean_data()
    cleaned_cols = processor.data.shape[1]
    assert cleaned_cols == original_cols - 1

'''Verifica que la variable objetivo solo contenga 0 y 1, y sin nulos'''
def test_validate_target_variable(processor):
    processor.validate_target_variable()
    target_col = processor.data.columns[-1]
    assert processor.data[target_col].isin([0, 1]).all()
    assert processor.data[target_col].isnull().sum() == 0

'''Verifica que el método load_clean_data cree una copia exacta del DataFrame limpio'''
def test_load_clean_data(processor):
    processor.load_clean_data()
    assert processor.cleaned_data is not None
    assert isinstance(processor.cleaned_data, pd.DataFrame)
    pd.testing.assert_frame_equal(processor.cleaned_data, processor.data)

'''Verifica que los datos se exporten correctamente a un archivo CSV'''
def test_export_data(processor):
    processor.clean_data()
    processor.export_data()
    exported = pd.read_csv(processor.output_path)
    assert not exported.empty

.....                                                                                        [100%]
5 passed in 5.37s
